In [ ]:
# ---------------- 1. GEREKLİ KÜTÜPHANELERİN KURULUMU ----------------
!pip install gradio transformers peft bitsandbytes accelerate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.9 MB/s eta 0:00:00


In [ ]:
# ---------------- 2. KÜTÜPHANELER ----------------
import gc
import torch
import gradio as gr

from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel


In [ ]:
# ---------------- 3. HUGGING FACE GİRİŞİ ----------------
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('hf')
login(token=hf_token)


In [ ]:
# ---------------- 4. ETİKETLER VE MODEL BİLGİLERİ ----------------

LABELS = ["igrenme", "korku", "minnet", "pismanlik", "mutluluk", "ofke", "saskinlik", "uzuntu"]

# Hazır eğitilmiş modellerin Hugging Face adresleri.
# Kendi modelinizi kullanmak isterseniz bu alanları değiştirebilirsiniz.
MODEL_INFO = {
    "BERTurk": {
        "type": "slm",
        "model_id": "nihalenc/berturk-duygu-analizi",
        "max_length": 128
    },
    "ELECTRA": {
        "type": "slm",
        "model_id": "nihalenc/electra-duygu-analizi",
        "max_length": 128
    },
    "LLaMA-3": {
        "type": "llm_lora",
        "base_model_id": "meta-llama/Meta-Llama-3-8B",
        "adapter_id": "nihalenc/llama3-duygu-analizi-lora",
        "max_length": 128
    },
    "Mistral": {
        "type": "llm_lora",
        "base_model_id": "mistralai/Mistral-7B-v0.3",
        "adapter_id": "byzhsm/mistral3",
        "max_length": 128
    }
}


In [ ]:
# ---------------- 5. MODEL ÖNBELLEĞİ VE BELLEK TEMİZLEME ----------------
# Aynı anda yalnızca seçili model bellekte tutulur.
current_model_name = None
current_model = None
current_tokenizer = None
current_max_length = 128

def clear_memory():
    global current_model, current_tokenizer, current_model_name

    current_model = None
    current_tokenizer = None
    current_model_name = None

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# ---------------- 6. SEÇİLEN MODELİ YÜKLEME ----------------
def load_selected_model(model_name):
    global current_model_name, current_model, current_tokenizer, current_max_length

    # Seçili model zaten yüklüyse tekrar yüklenmez.
    if current_model_name == model_name and current_model is not None:
        return current_model, current_tokenizer, current_max_length

    clear_memory()

    info = MODEL_INFO[model_name]
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # SLM modeller doğrudan yüklenir.
    if info["type"] == "slm":
        tokenizer = AutoTokenizer.from_pretrained(info["model_id"])
        model = AutoModelForSequenceClassification.from_pretrained(info["model_id"])
        model.to(device)

    else:
        if not torch.cuda.is_available():
            raise RuntimeError("LLaMA/Mistral için GPU gerekli. Colab Runtime > Change runtime type > GPU seçin.")

        # LLM modeller bellek tasarrufu için 4-bit yüklenir.
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        tokenizer = AutoTokenizer.from_pretrained(info["base_model_id"])

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        base_model = AutoModelForSequenceClassification.from_pretrained(
            info["base_model_id"],
            num_labels=len(LABELS),
            quantization_config=bnb_config,
            device_map="auto"
        )

        base_model.config.pad_token_id = tokenizer.pad_token_id

        # Fine-tune edilmiş LoRA adaptörü base modele eklenir.
        model = PeftModel.from_pretrained(
            base_model,
            info["adapter_id"]
        )

    model.eval()

    current_model_name = model_name
    current_model = model
    current_tokenizer = tokenizer
    current_max_length = info["max_length"]

    return model, tokenizer, current_max_length


In [ ]:
# ---------------- 7. DUYGU TAHMİN FONKSİYONU ----------------
def predict_emotion(text, model_name):
    if not text or text.strip() == "":
        return "Metin boş olamaz.", {}

    try:
        model, tokenizer, max_length = load_selected_model(model_name)

        device = next(model.parameters()).device

        # Girilen metin modelin beklediği formata çevrilir.
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)[0]

        pred_id = torch.argmax(probs).item()
        pred_label = LABELS[pred_id]
        confidence = float(probs[pred_id])

        # Tüm sınıflar için olasılık değerleri hazırlanır.
        prob_dict = {
            LABELS[i]: float(probs[i])
            for i in range(len(LABELS))
        }

        result_text = f"Tahmin: {pred_label} | Güven: {confidence:.3f}"

        return result_text, prob_dict

    except Exception as e:
        return f"Hata: {str(e)}", {}


In [ ]:
# ---------------- 8. GRADIO ARAYÜZÜ ----------------
# Kullanıcı metin ve model seçer, sonuç arayüzde gösterilir.
demo = gr.Interface(
    fn=predict_emotion,
    inputs=[
        gr.Textbox(lines=5, label="Metin giriniz"),
        gr.Dropdown(
            choices=["BERTurk", "ELECTRA", "LLaMA-3", "Mistral"],
            value="BERTurk",
            label="Model seçiniz"
        )
    ],
    outputs=[
        gr.Textbox(label="Tahmin Sonucu"),
        gr.Label(label="Duygu Olasılıkları")
    ],
    title="Türkçe Duygu Analizi",
    description="BERTurk, ELECTRA, LLaMA-3 ve Mistral modelleri ile 8 sınıflı duygu tahmini"
)

demo.launch(share=True)
